# Understanding the PANDA data — exploratory analysis & label quality

Before trusting any model, we need to understand the data: how the target is
distributed, how the two contributing centres differ, and most importantly for
critically interpreting our results **where the labels themselves are inconsistent
or noisy**. The PANDA organisers state plainly that the labels are imperfect and that
expert pathologists often disagree, so quantifying that is part of the science, not a
footnote.

This notebook is organised as:
1. The target (ISUP grade) and its imbalance
2. The two centres (Radboud vs Karolinska) — a built-in domain shift
3. **Gleason to ISUP consistency** — detecting labelling inaccuracies directly
4. Visual inspection of slides across grades
5. Tissue content, pen marks, and train/test artefacts
6. Segmentation masks — two different annotation schemes
7. Implications for modelling and error analysis (false positives / negatives)



In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
PAL = sns.color_palette('viridis', 6)
try:
    import openslide; HAVE_OS = True
except Exception:
    HAVE_OS = False

DATA = '/kaggle/input/competitions/prostate-cancer-grade-assessment'
IMG  = os.path.join(DATA, 'train_images')
MSK  = os.path.join(DATA, 'train_label_masks')
df = pd.read_csv(os.path.join(DATA, 'train.csv'))
print('openslide:', HAVE_OS, '| rows:', len(df))
print('missing values:\n', df.isna().sum())
df.head()

## 1. The target: ISUP grade (0–5)

ISUP grade is the clinical severity score we must predict. **0 = benign** (no cancer),
**1–5 = increasing aggressiveness**. Two things to read off the distribution:
* whether the classes are **imbalanced** (they are — benign and low grades dominate),
  which is why a plain accuracy would be misleading and the challenge uses
  **quadratic weighted kappa (QWK)**, which rewards being close on the ordinal scale
* the fact that the grades are **ordered**, predicting 4 when the truth is 5 is a far
  smaller error than predicting 0. This ordering motivates the ordinal model head.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
vc = df.isup_grade.value_counts().sort_index()
bars = ax[0].bar(vc.index, vc.values, color=PAL)
for b, v in zip(bars, vc.values):
    ax[0].text(b.get_x()+b.get_width()/2, v, f'{v}\n{v/len(df)*100:.1f}%',
               ha='center', va='bottom', fontsize=9)
ax[0].set_title('ISUP grade distribution'); ax[0].set_xlabel('ISUP grade'); ax[0].set_ylabel('# slides')
ax[0].set_ylim(0, vc.max()*1.15)

ax[1].pie(vc.values, labels=[f'ISUP {i}' for i in vc.index], colors=PAL,
          autopct='%1.1f%%', startangle=90, wedgeprops=dict(width=0.45))
ax[1].set_title('Share of each grade')
plt.tight_layout(); plt.show()

## 2. Two centres: Radboud vs Karolinska

The data comes from **two institutions** using **different scanners and different
pathologists**. This is a built-in *domain shift*, a model can learn centre-specific
appearance (stain, scanner colour) instead of biology. If the two centres also have
**different grade distributions**, a model can even exploit that as a shortcut. We
check both. (This is the basis for our cross-centre generalisation experiment.)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
pc = df.data_provider.value_counts()
ax[0].bar(pc.index, pc.values, color=sns.color_palette('Set2', len(pc)))
for i, v in enumerate(pc.values): ax[0].text(i, v, str(v), ha='center', va='bottom')
ax[0].set_title('Slides per centre'); ax[0].set_ylabel('# slides')

ct = pd.crosstab(df.data_provider, df.isup_grade, normalize='index') * 100
ct.T.plot(kind='bar', ax=ax[1], color=sns.color_palette('Set2', 2))
ax[1].set_title('ISUP distribution by centre (%)'); ax[1].set_xlabel('ISUP grade')
ax[1].set_ylabel('% of that centre'); ax[1].legend(title='centre')
plt.tight_layout(); plt.show()
print(pd.crosstab(df.data_provider, df.isup_grade))

## 3. Gleason → ISUP consistency — detecting label inaccuracies

ISUP grade is **derived deterministically** from the Gleason score by fixed rules:

| Gleason | ISUP | | Gleason | ISUP |
|---|---|---|---|---|
| negative / 0+0 | 0 | | 4+4, 3+5, 5+3 | 4 |
| 3+3 | 1 | | 4+5, 5+4, 5+5 | 5 |
| 3+4 | 2 | | | |
| 4+3 | 3 | | | |

So for every slide we can compute the **expected** ISUP from its Gleason score and
compare it to the **recorded** ISUP. Any mismatch is a concrete **labelling
inaccuracy** exactly the kind of error the organisers warned about. Counting and
attributing these (which centre? which grades?) is direct evidence for our results
discussion.

In [ ]:
GLEASON_TO_ISUP = {
    'negative': 0, '0+0': 0,
    '3+3': 1, '3+4': 2, '4+3': 3,
    '4+4': 4, '3+5': 4, '5+3': 4,
    '4+5': 5, '5+4': 5, '5+5': 5,
}
print('gleason_score values:')
print(df.gleason_score.value_counts())

# annotated crosstab: Gleason score vs recorded ISUP
ct = pd.crosstab(df.gleason_score, df.isup_grade)
plt.figure(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': '# slides'})
plt.title('Gleason score vs recorded ISUP grade'); plt.xlabel('recorded ISUP'); plt.ylabel('Gleason score')
plt.tight_layout(); plt.show()

In [ ]:
df['expected_isup'] = df.gleason_score.map(GLEASON_TO_ISUP)
unknown = df.expected_isup.isna().sum()
df_known = df.dropna(subset=['expected_isup']).copy()
df_known['expected_isup'] = df_known.expected_isup.astype(int)
df_known['mismatch'] = df_known.expected_isup != df_known.isup_grade

n_mis = int(df_known.mismatch.sum())
print(f'rows with unmappable gleason_score : {unknown}')
print(f'rows where recorded ISUP != expected: {n_mis} ({n_mis/len(df_known)*100:.2f}%)')
print('\nmismatches by centre:')
print(df_known[df_known.mismatch].data_provider.value_counts())
print('\nexample mismatched rows:')
cols = ['image_id', 'data_provider', 'gleason_score', 'isup_grade', 'expected_isup']
display(df_known[df_known.mismatch][cols].head(10))

In [ ]:
# where do mismatches sit? recorded vs expected ISUP for the inconsistent rows
mm = df_known[df_known.mismatch]
if len(mm):
    cm = pd.crosstab(mm.expected_isup, mm.isup_grade)
    plt.figure(figsize=(6.5, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar_kws={'label': '# mismatched slides'})
    plt.title('Label inconsistencies: expected vs recorded ISUP')
    plt.xlabel('recorded ISUP'); plt.ylabel('expected ISUP (from Gleason)')
    plt.tight_layout(); plt.show()
else:
    print('No mismatches found under this mapping.')

## 4. Visual inspection — what does each grade look like?

Numbers only go so far, we should see the tissue. Below is one example whole-slide
thumbnail per ISUP grade. Higher grades show more disrupted, fused glandular
architecture. This also exposes how variable the slides are in size, tissue amount and
staining, variability the model must cope with.

In [ ]:
def thumb(image_id, size=512):
    s = openslide.OpenSlide(os.path.join(IMG, f'{image_id}.tiff'))
    t = np.asarray(s.get_thumbnail((size, size)).convert('RGB')); s.close()
    return t

if HAVE_OS:
    fig, ax = plt.subplots(1, 6, figsize=(20, 5))
    for g in range(6):
        sub = df[df.isup_grade == g]
        row = sub.sample(1, random_state=1).iloc[0]
        ax[g].imshow(thumb(row.image_id)); ax[g].axis('off')
        ax[g].set_title(f'ISUP {g}\n{row.data_provider}\nGleason {row.gleason_score}', fontsize=9)
    plt.suptitle('One example whole-slide image per ISUP grade', y=1.02); plt.show()
else:
    print('openslide unavailable — skipping image previews')

## 5. Tissue content, pen marks, and a train/test artefact

Two practical data issues:
* **Tissue fraction varies a lot.** Some biopsies are mostly white background, so the
  tile-selection step (keep the most-tissue tiles) is essential. Slides with very
  little tissue are intrinsically harder.
* **Pen marks.** The organisers note the *training* slides sometimes carry stray pen
  marks but the *test* slides do not — a genuine train/test distribution difference a
  model could accidentally key on.

In [ ]:
if HAVE_OS:
    sample = df.sample(min(300, len(df)), random_state=0)
    fracs = []
    for iid in sample.image_id:
        t = thumb(iid, 256).astype(np.float32)
        gray = t.mean(2)
        fracs.append(float((gray < 220).mean()))   # darker than near-white = tissue
    sample = sample.assign(tissue_frac=fracs)

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    sns.histplot(sample.tissue_frac, bins=30, ax=ax[0], color='#4C72B0')
    ax[0].set_title('Tissue fraction (sample of 300 slides)'); ax[0].set_xlabel('fraction of non-background pixels')
    sns.boxplot(data=sample, x='isup_grade', y='tissue_frac', ax=ax[1], palette='viridis')
    ax[1].set_title('Tissue fraction by ISUP grade'); ax[1].set_xlabel('ISUP grade')
    plt.tight_layout(); plt.show()
    print('lowest-tissue slides (hardest to tile):')
    display(sample.nsmallest(5, 'tissue_frac')[['image_id','data_provider','isup_grade','tissue_frac']])

## 6. Segmentation masks — two different annotation schemes

Some training slides come with pixel masks, but the **two centres label them
differently**, which is itself a data-understanding (and consistency) issue:

* **Radboud** — glands labelled individually: 0 background, 1 stroma, 2 benign
  epithelium, **3/4/5 = cancerous epithelium of Gleason pattern 3/4/5**.
* **Karolinska** — coarser regions: 0 background, 1 benign, **2 cancer** (no Gleason
  pattern detail).

So a Radboud mask carries far more information than a Karolinska one. We first check
how many slides even have masks, then overlay one example from each centre.

In [ ]:
mask_ids = set(f.replace('_mask.tiff', '') for f in os.listdir(MSK)) if os.path.exists(MSK) else set()
df['has_mask'] = df.image_id.isin(mask_ids)
print('slides with a mask:', int(df.has_mask.sum()), 'of', len(df))
print(df.groupby('data_provider').has_mask.mean().mul(100).round(1).astype(str) + ' %')

In [ ]:
def read_level0_small(path, size=512):
    s = openslide.OpenSlide(path); lv = s.level_count - 1
    arr = np.asarray(s.read_region((0, 0), lv, s.level_dimensions[lv]).convert('RGB')); s.close()
    return arr

RAD_CMAP = mcolors.ListedColormap(['white','lightgray','#7fbf7b','#fee08b','#fc8d59','#d73027'])
KAR_CMAP = mcolors.ListedColormap(['white','#7fbf7b','#d73027'])

def show_with_mask(image_id, provider):
    img  = read_level0_small(os.path.join(IMG, f'{image_id}.tiff'))
    mask = read_level0_small(os.path.join(MSK, f'{image_id}_mask.tiff'))[..., 0]  # label in channel 0
    cmap = RAD_CMAP if provider == 'radboud' else KAR_CMAP
    vmax = 5 if provider == 'radboud' else 2
    fig, ax = plt.subplots(1, 2, figsize=(11, 6))
    ax[0].imshow(img); ax[0].set_title(f'{provider} slide ({image_id[:8]})'); ax[0].axis('off')
    ax[1].imshow(mask, cmap=cmap, vmin=0, vmax=vmax, interpolation='nearest')
    ax[1].set_title('mask (label = channel 0)'); ax[1].axis('off')
    plt.tight_layout(); plt.show()

if HAVE_OS and mask_ids:
    for prov in ['radboud', 'karolinska']:
        sub = df[(df.data_provider == prov) & (df.has_mask) & (df.isup_grade >= 3)]
        if len(sub):
            show_with_mask(sub.sample(1, random_state=2).iloc[0].image_id, prov)

Notice the Radboud mask uses up to five tissue classes (you can see Gleason 3/4/5
regions in different colours), whereas the Karolinska mask only distinguishes
benign vs cancer. Any model or analysis that uses masks must treat the two schemes
separately and the asymmetry is one more reason the two centres are not
interchangeable.

## 7. Implications for modelling and error analysis


### Extending to false-positive / false-negative analysis


In [ ]:
# load your real validation predictions

from sklearn.metrics import cohen_kappa_score, confusion_matrix
val_df = pd.read_csv('/kaggle/input/datasets/shashaboii/val-effnet/val_df_effnet.csv')   # or '../input/<dataset>/val_df_effnet.csv'

# merge with the label-consistency info computed earlier in this notebook
m = val_df.merge(
    df_known[['image_id', 'gleason_score', 'expected_isup', 'mismatch']],
    on='image_id', how='left')
m['abs_err'] = (m.true_isup - m.pred_isup).abs()

print('validation slides analysed:', len(m))
print('overall QWK:',
      cohen_kappa_score(m.true_isup, m.pred_isup, weights='quadratic').round(4))

# 1) are our biggest errors actually the noisy-label slides?
worst = m.sort_values('abs_err', ascending=False).head(30)
print(f'\\nshare of 30 worst errors that are LABEL mismatches: {worst.mismatch.mean():.1%}')

# 2) clinically meaningful errors
fn = m[(m.pred_isup == 0) & (m.true_isup > 0)]   # missed cancer (predicted benign) - dangerous
fp = m[(m.pred_isup > 0) & (m.true_isup == 0)]   # false alarm (predicted cancer)
print(f'missed cancers (pred benign, truly cancer): {len(fn)}')
print(f'false alarms  (pred cancer, truly benign): {len(fp)}')

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
cm = confusion_matrix(m.true_isup, m.pred_isup, labels=list(range(6)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0])
ax[0].set_title('EffNet validation confusion'); ax[0].set_xlabel('predicted'); ax[0].set_ylabel('true')
sns.histplot(m.abs_err, bins=range(7), ax=ax[1], color='#4C72B0')
ax[1].set_title('Absolute error (|true − pred|)'); ax[1].set_xlabel('grades off')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

# load both models' validation predictions
eff = pd.read_csv('/kaggle/input/datasets/shashaboii/val-effnet/val_df_effnet.csv')      # or ../input/<dataset>/...
uni = pd.read_csv('/kaggle/input/datasets/shashaboii/val-df-effnet/val_df_uni.csv')  # fix path

def summary(d, name):
    qwk = cohen_kappa_score(d.true_isup, d.pred_isup, weights='quadratic')
    fn = int(((d.pred_isup == 0) & (d.true_isup > 0)).sum())   # missed cancer (dangerous)
    fp = int(((d.pred_isup > 0) & (d.true_isup == 0)).sum())   # false alarm
    qk = cohen_kappa_score(*[d[d.data_provider=='karolinska'][c] for c in ['true_isup','pred_isup']], weights='quadratic')
    qr = cohen_kappa_score(*[d[d.data_provider=='radboud'][c]    for c in ['true_isup','pred_isup']], weights='quadratic')
    return {'model': name, 'QWK': round(qwk,4), 'QWK_Karolinska': round(qk,4),
            'QWK_Radboud': round(qr,4), 'missed_cancers': fn, 'false_alarms': fp}

table = pd.DataFrame([summary(eff, 'EfficientNet-B0'), summary(uni, 'UNI + MIL')])
display(table)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, (d, name) in zip(ax, [(eff,'EfficientNet-B0'), (uni,'UNI + MIL')]):
    cm = confusion_matrix(d.true_isup, d.pred_isup, labels=list(range(6)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=a)
    a.set_title(name); a.set_xlabel('predicted'); a.set_ylabel('true')
plt.tight_layout(); plt.show()